In [ ]:
!pip install pytorch_forecasting

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet, QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
from pytorch_lightning import LightningModule
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Скриптата како inputs го зима тренинраниот модел, најдобрата епоха и датасетот со кој е трениран и според тоа генереира
#нов датасет со buy/sell/hold сигнали.

#Strong Signal (≥0.15% difference): High confidence BUY/SELL
#Weak Signal (0.05-0.15% difference): Moderate confidence BUY/SELL
#Hold Signal (<0.05% difference): Low confidence, stay neutral

import pandas as pd
import numpy as np
import torch
import os
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, QuantileLoss
from pytorch_forecasting.data import GroupNormalizer


def safe_torch_load(path):
    """Safely load torch files with weights_only=False"""
    try:
        return torch.load(path, map_location='cpu', weights_only=False)
    except Exception as e1:
        try:
            # Add safe globals for Lightning
            torch.serialization.add_safe_globals(['lightning.fabric.utilities.data.AttributeDict'])
            return torch.load(path, map_location='cpu', weights_only=True)
        except Exception as e2:
            print(f"Both loading methods failed: {e1}, {e2}")
            return None

def create_prediction_dataset(data, max_encoder_length=216, max_prediction_length=48):
    """
    Create a proper TimeSeriesDataSet for prediction from your data with better error handling
    """
    print(f"🔨 Creating prediction dataset with encoder length {max_encoder_length}...")

    try:
        # Check if we have enough data
        min_required_length = max_encoder_length + max_prediction_length + 10
        if len(data) < min_required_length:
            print(f"⚠️  Not enough data! Need at least {min_required_length}, got {len(data)}")
            print(f"🔧 Reducing encoder length to fit available data...")

            # Calculate maximum possible encoder length
            available_length = len(data) - max_prediction_length - 10
            max_encoder_length = min(max_encoder_length, max(available_length, 50))  # Minimum 50 for TFT
            print(f"📏 Using reduced encoder length: {max_encoder_length}")

        # Use larger portion of data for prediction to ensure we have enough samples
        data_portion = max(500, min_required_length + 100)  # Use at least 500 or required + buffer
        recent_data = data.tail(data_portion).copy().reset_index(drop=True)

        print(f"📊 Using {len(recent_data)} data points for prediction dataset")

        # Ensure time_idx is continuous and starts from 0
        recent_data['time_idx'] = range(len(recent_data))

        # Ensure group column exists
        if 'group' not in recent_data.columns:
            recent_data['group'] = 'AAPL'

        # Get all numeric columns for time-varying reals
        numeric_cols = recent_data.select_dtypes(include=[np.number]).columns.tolist()

        # Remove time_idx from time-varying variables
        time_varying_reals = [col for col in numeric_cols if col not in ['time_idx']]

        # Ensure we have close price
        if 'close' not in time_varying_reals:
            print("❌ 'close' column not found in data!")
            return None

        # Limit to important features to avoid complexity - but be flexible with what's available
        important_features = ['close', 'EMA', 'SMA', 'volatility_50', 'momentum_1', 'roc_1',
                            'volume', 'high', 'low', 'open']

        available_features = [col for col in important_features if col in time_varying_reals]

        # If we don't have enough features, use what we have
        if len(available_features) < 3:
            print(f"⚠️  Only {len(available_features)} important features found, using all numeric columns...")
            time_varying_reals = time_varying_reals[:10]  # Limit to first 10 to avoid issues
        else:
            time_varying_reals = available_features

        # Ensure close is always included
        if 'close' not in time_varying_reals:
            time_varying_reals.append('close')

        print(f"📊 Using features ({len(time_varying_reals)}): {time_varying_reals}")

        # Check for NaN values and handle them
        for col in time_varying_reals:
            if recent_data[col].isna().sum() > 0:
                print(f"⚠️  Found {recent_data[col].isna().sum()} NaN values in {col}, filling with forward fill...")
                recent_data[col] = recent_data[col].fillna(method='ffill').fillna(method='bfill')

        # Create dataset with more robust parameters
        try:
            dataset = TimeSeriesDataSet(
                recent_data,
                time_idx="time_idx",
                target="close",
                group_ids=["group"],
                max_encoder_length=max_encoder_length,
                max_prediction_length=max_prediction_length,
                time_varying_unknown_reals=time_varying_reals,
                target_normalizer=GroupNormalizer(groups=["group"], transformation="softplus"),
                add_relative_time_idx=True,
                add_target_scales=True,
                randomize_length=None,
                allow_missing_timesteps=True,  # Allow missing timesteps
            )

            print(f"✅ Dataset created successfully: {len(dataset)} samples")

            # Verify dataset has samples
            if len(dataset) == 0:
                print("❌ Dataset has 0 samples! This usually means:")
                print("  - Not enough data for the encoder length")
                print("  - Data format issues")
                print("  - Missing required columns")
                return None

            return dataset

        except Exception as dataset_error:
            print(f"❌ Dataset creation failed: {dataset_error}")

            # Try with even smaller encoder length
            if max_encoder_length > 20:
                print("🔧 Trying with smaller encoder length...")
                return create_prediction_dataset(data, max_encoder_length=20, max_prediction_length=max_prediction_length)
            else:
                print("❌ Cannot create dataset even with minimum encoder length")
                return None

    except Exception as e:
        print(f"❌ Dataset creation failed with error: {e}")
        print(f"📊 Data shape: {data.shape if data is not None else 'None'}")
        print(f"📊 Data columns: {list(data.columns) if data is not None else 'None'}")
        return None

def load_model_with_proper_reconstruction(checkpoint_path, pytorch_path, sample_data, encoder_length=216, prediction_length=48):
    """
    Load model with proper reconstruction using sample data
    """
    model = None
    model_name = ""
    prediction_dataset = None


    print(f"🔨 Creating prediction dataset for model loading...")
    prediction_dataset = create_prediction_dataset(sample_data, max_encoder_length=encoder_length, max_prediction_length=prediction_length)

    if prediction_dataset is None:
        print("❌ Could not create prediction dataset - trying with different parameters...")
        # Try with smaller encoder lengths but keep prediction length
        for smaller_length in [150, 100, 72]:
            print(f"🔧 Trying with encoder length {smaller_length}...")
            prediction_dataset = create_prediction_dataset(sample_data, max_encoder_length=smaller_length, max_prediction_length=prediction_length)
            if prediction_dataset is not None:
                encoder_length = smaller_length
                print(f"✅ Successfully created dataset with encoder length {encoder_length}")
                break

        if prediction_dataset is None:
            print("❌ Could not create prediction dataset with any encoder length!")
            return None, "", None

    # Method 1: Try PyTorch state dict with proper reconstruction
    if pytorch_path and os.path.exists(pytorch_path):
        print(f"🔥 Loading PyTorch model: {pytorch_path}")
        try:
            state_dict = safe_torch_load(pytorch_path)

            if isinstance(state_dict, dict) and not hasattr(state_dict, 'forward'):
                print("📋 Found state dict - creating proper TFT model...")

                # Create TFT model with same structure as training
                model = TemporalFusionTransformer.from_dataset(
                    prediction_dataset,
                    learning_rate=0.03,
                    hidden_size=64,  # Adjust based on your training config
                    attention_head_size=4,
                    dropout=0.1,
                    hidden_continuous_size=32,
                    output_size=7,  # Quantiles
                    loss=QuantileLoss(),
                    reduce_on_plateau_patience=4,
                )

                # Load state dict
                model.load_state_dict(state_dict, strict=False)
                model.eval()
                model_name = "reconstructed_from_state_dict"
                print("✅ Model reconstructed successfully!")
                return model, model_name, prediction_dataset

            elif hasattr(state_dict, 'forward'):
                # It's a complete model
                model = state_dict
                if hasattr(model, 'tft_model'):
                    model = model.tft_model
                model.eval()
                model_name = "complete_pytorch_model"
                return model, model_name, prediction_dataset

        except Exception as e:
            print(f"❌ PyTorch loading failed: {e}")

    # Method 2: Try checkpoint with fixed loading
    if checkpoint_path and os.path.exists(checkpoint_path):
        try:
            print("📦 Trying checkpoint with proper loading...")
            checkpoint = safe_torch_load(checkpoint_path)

            if checkpoint:
                # Try to create model and load weights
                model = TemporalFusionTransformer.from_dataset(
                    prediction_dataset,
                    learning_rate=0.03,
                    hidden_size=64,
                    attention_head_size=4,
                    dropout=0.1,
                    hidden_continuous_size=32,
                    output_size=7,
                    loss=QuantileLoss(),
                )

                # Try different state dict keys
                if 'tft_model_state_dict' in checkpoint:
                    model.load_state_dict(checkpoint['tft_model_state_dict'], strict=False)
                elif 'state_dict' in checkpoint:
                    # Extract TFT state from wrapper
                    tft_state = {k.replace('tft_model.', ''): v for k, v in checkpoint['state_dict'].items()
                                if k.startswith('tft_model.')}
                    if tft_state:
                        model.load_state_dict(tft_state, strict=False)
                    else:
                        model.load_state_dict(checkpoint['state_dict'], strict=False)

                model.eval()
                model_name = "checkpoint_reconstructed"
                return model, model_name, prediction_dataset

        except Exception as e:
            print(f"❌ Checkpoint loading failed: {e}")

    return None, "", prediction_dataset

def generate_real_tft_predictions(model, dataset, num_predictions=100):
    """
    Generate real TFT predictions - FIXED for target extraction
    """
    print(f"🔮 Generating {num_predictions} real TFT predictions...")
    print(f"🔍 Model type: {type(model).__name__}")

    # Skip the broken predict() method, go directly to manual processing
    print("🔧 Using direct model forward pass (wrapper-compatible)...")

    try:
        predictions = []
        actuals = []
        indices = []

        # Create data loader for prediction with smaller batch size
        dataloader = dataset.to_dataloader(train=False, batch_size=1, num_workers=0)

        print(f"📊 Processing {len(dataloader)} batches...")

        count = 0
        for batch_idx, batch in enumerate(dataloader):
            if count >= num_predictions:
                break

            try:
                # Extract data from batch
                x, y = batch

                # Debug: Print shapes
                if batch_idx == 0:
                    print(f"🔍 Input shapes: x={[v.shape if torch.is_tensor(v) else type(v) for v in x.values()]}")
                    print(f"🔍 Target shape: y={y.shape if torch.is_tensor(y) else type(y)}")

                # FIXED: Better target extraction handling tuple case
                actual_value = None
                if isinstance(y, tuple):
                    # Handle tuple case - try different elements
                    for i, y_element in enumerate(y):
                        if torch.is_tensor(y_element):
                            try:
                                if y_element.numel() == 1:
                                    actual_value = float(y_element.item())
                                    if batch_idx == 0:
                                        print(f"🔍 Using tuple element {i} as actual value")
                                    break
                                elif y_element.numel() > 1:
                                    # Take the first element if multi-element tensor
                                    actual_value = float(y_element.flatten()[0].item())
                                    if batch_idx == 0:
                                        print(f"🔍 Using first element of tuple element {i} as actual value")
                                    break
                            except Exception as e:
                                if batch_idx == 0:
                                    print(f"⚠️  Could not extract from tuple element {i}: {e}")
                                continue
                elif torch.is_tensor(y):
                    # Handle tensor case
                    if y.numel() == 1:
                        actual_value = float(y.item())
                    elif y.numel() > 1:
                        # Take the first element or try to extract the first prediction
                        if y.dim() > 1:
                            actual_value = float(y[0][0].item())
                        else:
                            actual_value = float(y[0].item())
                else:
                    # Handle other cases
                    try:
                        actual_value = float(y)
                    except:
                        actual_value = float(y[0]) if hasattr(y, '__getitem__') else None

                if actual_value is None:
                    if batch_idx < 5:
                        print(f"⚠️  Could not extract actual value from batch {batch_idx}")
                    continue

                # Make prediction using direct forward pass
                with torch.no_grad():
                    try:
                        # Try multiple approaches for wrapper models
                        model_output = None

                        # Method 1: Direct call
                        try:
                            model_output = model(x)
                            if batch_idx == 0:
                                print(f"✅ Direct model call succeeded")
                        except Exception as e1:
                            if batch_idx == 0:
                                print(f"❌ Direct call failed: {e1}")

                        # Method 2: Check if it's a wrapper with tft_model
                        if model_output is None and hasattr(model, 'tft_model'):
                            try:
                                model_output = model.tft_model(x)
                                if batch_idx == 0:
                                    print(f"✅ Wrapper.tft_model call succeeded")
                            except Exception as e2:
                                if batch_idx == 0:
                                    print(f"❌ Wrapper call failed: {e2}")

                        # Method 3: Check for forward method
                        if model_output is None and hasattr(model, 'forward'):
                            try:
                                model_output = model.forward(x)
                                if batch_idx == 0:
                                    print(f"✅ Forward method call succeeded")
                            except Exception as e3:
                                if batch_idx == 0:
                                    print(f"❌ Forward method failed: {e3}")

                        # Method 4: Try to access underlying model
                        if model_output is None:
                            for attr_name in ['model', 'tft', 'net', '_model']:
                                if hasattr(model, attr_name):
                                    try:
                                        sub_model = getattr(model, attr_name)
                                        model_output = sub_model(x)
                                        if batch_idx == 0:
                                            print(f"✅ Sub-model {attr_name} call succeeded")
                                        break
                                    except Exception as e4:
                                        if batch_idx == 0:
                                            print(f"❌ Sub-model {attr_name} failed: {e4}")
                                        continue

                        if model_output is None:
                            if batch_idx == 0:
                                print("❌ All model call methods failed")
                            continue

                        # Debug output structure on first batch
                        if batch_idx == 0:
                            print(f"🔍 Model output type: {type(model_output)}")
                            if hasattr(model_output, 'prediction'):
                                print(f"🔍 Has prediction attribute")
                            if isinstance(model_output, dict):
                                print(f"🔍 Output keys: {list(model_output.keys())}")
                            if torch.is_tensor(model_output):
                                print(f"🔍 Output shape: {model_output.shape}")

                        # Extract prediction from model output
                        prediction_tensor = None

                        if hasattr(model_output, 'prediction'):
                            prediction_tensor = model_output.prediction
                        elif isinstance(model_output, dict):
                            # Try different common keys
                            for key in ['prediction', 'output', 'logits', 'pred', 'y_hat']:
                                if key in model_output:
                                    prediction_tensor = model_output[key]
                                    break
                        elif torch.is_tensor(model_output):
                            prediction_tensor = model_output
                        else:
                            if batch_idx == 0:
                                print(f"❌ Unknown output format: {type(model_output)}")
                            continue

                        if prediction_tensor is None:
                            if batch_idx == 0:
                                print("❌ Could not extract prediction tensor")
                            continue

                        # Convert tensor to numpy
                        if torch.is_tensor(prediction_tensor):
                            pred_array = prediction_tensor.detach().cpu().numpy()
                        else:
                            pred_array = prediction_tensor

                        if batch_idx == 0:
                            print(f"🔍 Prediction array shape: {pred_array.shape}")

                        # Extract prediction value based on shape
                        predicted_value = None

                        if pred_array.ndim == 3:  # [batch, time, quantiles]
                            if pred_array.shape[1] == 48:  # 48 time steps prediction
                                # Use the first time step prediction for trading signal
                                if pred_array.shape[-1] == 7:
                                    predicted_value = float(pred_array[0, 0, 3])  # Median quantile, first time step
                                else:
                                    predicted_value = float(pred_array[0, 0, 0])  # First time step
                            elif pred_array.shape[-1] == 7:
                                predicted_value = float(pred_array[0, 0, 3])  # Median quantile
                            else:
                                predicted_value = float(pred_array[0, 0, 0])
                        elif pred_array.ndim == 2:  # [batch, quantiles] or [batch, time]
                            if pred_array.shape[-1] == 7:
                                predicted_value = float(pred_array[0, 3])  # Median quantile
                            elif pred_array.shape[-1] == 48:
                                predicted_value = float(pred_array[0, 0])  # First time step
                            else:
                                predicted_value = float(pred_array[0, 0])
                        elif pred_array.ndim == 1:  # [batch]
                            predicted_value = float(pred_array[0])
                        else:
                            predicted_value = float(pred_array.flatten()[0])

                        if predicted_value is not None:
                            predictions.append(predicted_value)
                            actuals.append(actual_value)
                            indices.append(batch_idx)
                            count += 1

                            if count <= 10 or count % 20 == 0:  # Show first 10 and every 20th
                                print(f"  ✅ Prediction {count}: Actual=${actual_value:.2f}, Predicted=${predicted_value:.2f}")

                    except Exception as forward_error:
                        if batch_idx < 5:  # Only show first few errors
                            print(f"⚠️  Forward pass {batch_idx} failed: {forward_error}")
                        continue

            except Exception as batch_error:
                if batch_idx < 5:  # Only show first few errors
                    print(f"⚠️  Batch {batch_idx} failed: {batch_error}")
                continue

        if predictions:
            print(f"✅ Generated {len(predictions)} predictions using direct forward pass")
            return predictions, actuals, indices
        else:
            print("❌ No predictions generated - all methods failed")

    except Exception as main_error:
        print(f"❌ Main prediction process failed: {main_error}")

    return None, None, None

possible_data_paths = [
    "/content/sample_data/AAPL_FIXED_clean.csv",
    "AAPL_FIXED_clean.csv",
    "/content/AAPL_FIXED_clean.csv",
    "./AAPL_FIXED_clean.csv"
]

print(f"📂 Looking for data file...")
data_file = None
df = None

for path in possible_data_paths:
    if os.path.exists(path):
        data_file = path
        print(f"✅ Found data at: {path}")
        try:
            df = pd.read_csv(path)
            print(f"✅ Data loaded: {df.shape}")
            print(f"First few columns: {list(df.columns)[:10]}")
            break
        except Exception as e:
            print(f"❌ Error loading {path}: {e}")
            continue

if df is None:
    print("❌ Could not find or load data file!")
    print("📁 Available files in current directory:")
    try:
        for file in os.listdir('.'):
            if file.endswith('.csv'):
                print(f"  - {file}")
    except:
        pass
    exit()

possible_checkpoint_paths = [
    "/content/best-tft-epoch=45.ckpt",
    "/content/sample_data/best-tft-epoch=45.ckpt"
]

possible_pytorch_paths = [
    "/content/trained_tft_model.pt",
    "/content/sample_data/trained_tft_model.pt"
]

checkpoint_file = None
pytorch_file = None

for path in possible_checkpoint_paths:
    if os.path.exists(path):
        checkpoint_file = path
        break

for path in possible_pytorch_paths:
    if os.path.exists(path):
        pytorch_file = path
        break

print(f"\n🔍 Loading model files...")
print(f"  Checkpoint: {checkpoint_file if checkpoint_file else 'Not found'}")
print(f"  PyTorch: {pytorch_file if pytorch_file else 'Not found'}")

encoder_lengths_to_try = [216, 150, 100, 72]
prediction_length = 48
model = None
model_name = ""
prediction_dataset = None

for encoder_length in encoder_lengths_to_try:
    print(f"\n🔧 Trying with encoder length: {encoder_length}, prediction length: {prediction_length}")
    model, model_name, prediction_dataset = load_model_with_proper_reconstruction(
        checkpoint_file, pytorch_file, df, encoder_length=encoder_length, prediction_length=prediction_length
    )

    if model is not None and prediction_dataset is not None:
        print(f"✅ Success with encoder length: {encoder_length}")
        break
    else:
        print(f"❌ Failed with encoder length: {encoder_length}")

if model is None or prediction_dataset is None:
    print("❌ Could not load model or create dataset with any encoder length!")
    print("💡 Possible issues:")
    print("  - Not enough data in the CSV file")
    print("  - Missing required columns (especially 'close')")
    print("  - Model file corruption")
    print("  - Data format issues")
    exit()

print(f"\n🎉 Successfully loaded {model_name} model!")
print(f"📊 Prediction dataset ready: {len(prediction_dataset)} samples")

# Get model info
try:
    param_count = sum(p.numel() for p in model.parameters())
    print(f"📈 Model parameters: {param_count:,}")
    print(f"📋 Model type: {type(model).__name__}")
except Exception as e:
    print(f"⚠️  Could not get model info: {e}")

# Step 3: Generate real predictions
print(f"\n🚀 Generating REAL TFT model predictions...")

predictions, actuals, indices = generate_real_tft_predictions(model, prediction_dataset, num_predictions=100)

if predictions and actuals:
    print(f"\n🎯 Processing {len(predictions)} real predictions...")

    # Generate trading signals from real predictions
    signals = []

    for i, (pred, actual, idx) in enumerate(zip(predictions, actuals, indices)):
        price_change = (pred - actual) / actual if actual != 0 else 0

        # THREE-SIGNAL STRATEGY: BUY/SELL/HOLD with different thresholds
        # Calculate the percentage difference between predicted and actual
        price_diff_pct = abs(price_change) * 100

        # Define thresholds for different signal strengths
        strong_signal_threshold = 0.15   # 0.15% - strong signal
        weak_signal_threshold = 0.05     # 0.05% - weak signal

        if pred > actual:
            # Prediction is higher than actual (bullish)
            if price_diff_pct >= strong_signal_threshold:
                signal = "BUY"
                signal_num = 1
                confidence = min(price_diff_pct / 0.5, 1.0)  # Strong confidence
            elif price_diff_pct >= weak_signal_threshold:
                signal = "BUY"
                signal_num = 1
                confidence = min(price_diff_pct / 0.3, 0.7)  # Moderate confidence
            else:
                signal = "HOLD"
                signal_num = 0
                confidence = 0.3  # Low confidence for small differences
        else:
            # Prediction is lower than actual (bearish)
            if price_diff_pct >= strong_signal_threshold:
                signal = "SELL"
                signal_num = -1
                confidence = min(price_diff_pct / 0.5, 1.0)  # Strong confidence
            elif price_diff_pct >= weak_signal_threshold:
                signal = "SELL"
                signal_num = -1
                confidence = min(price_diff_pct / 0.3, 0.7)  # Moderate confidence
            else:
                signal = "HOLD"
                signal_num = 0
                confidence = 0.3  # Low confidence for small differences

        # Ensure confidence is within reasonable bounds
        confidence = max(0.1, min(1.0, confidence))

        signals.append({
            'prediction_id': i + 1,
            'data_index': idx,
            'actual_price': actual,
            'predicted_price': pred,
            'price_change_pct': price_change * 100,
            'signal': signal,
            'signal_value': signal_num,
            'confidence': confidence
        })

    # Create results DataFrame
    signals_df = pd.DataFrame(signals)

    print(f"\n📈 REAL TFT TRADING SIGNALS COMPLETED")
    print("=" * 50)
    print(f"Total predictions: {len(signals_df)}")

    # Signal summary
    signal_counts = signals_df['signal'].value_counts()
    print(f"\n📊 Signal Distribution:")
    for signal in ['BUY', 'SELL', 'HOLD']:
        if signal in signal_counts:
            count = signal_counts[signal]
            emoji = {"BUY": "🟢", "SELL": "🔴", "HOLD": "⚪"}[signal]
            pct = count / len(signals_df) * 100
            print(f"  {emoji} {signal}: {count} ({pct:.1f}%)")

    # Show sample signals (first 20)
    print(f"\n🎯 SAMPLE REAL MODEL PREDICTIONS (First 20):")
    for idx, row in signals_df.head(20).iterrows():
        emoji = {"BUY": "🟢", "SELL": "🔴", "HOLD": "⚪"}[row['signal']]
        print(f"  {emoji} {row['signal']}: ${row['actual_price']:.2f} → ${row['predicted_price']:.2f} ({row['price_change_pct']:+.2f}%) [Conf: {row['confidence']:.2f}]")

    # Save results
    output_file = "real_tft_predictions.csv"
    signals_df.to_csv(output_file, index=False)
    print(f"\n💾 Real TFT predictions saved to: {output_file}")

    # Analysis
    buy_signals = signals_df[signals_df['signal'] == 'BUY']
    sell_signals = signals_df[signals_df['signal'] == 'SELL']

    if len(buy_signals) > 0:
        avg_buy_change = buy_signals['price_change_pct'].mean()
        max_buy_change = buy_signals['price_change_pct'].max()
        print(f"📈 BUY signals - Avg: {avg_buy_change:.2f}%, Max: {max_buy_change:.2f}%")

    if len(sell_signals) > 0:
        avg_sell_change = sell_signals['price_change_pct'].mean()
        min_sell_change = sell_signals['price_change_pct'].min()
        print(f"📉 SELL signals - Avg: {avg_sell_change:.2f}%, Min: {min_sell_change:.2f}%")

    # Calculate prediction accuracy
    mae = np.mean([abs(p - a) / a for p, a in zip(predictions, actuals) if a != 0])
    print(f"📊 Mean Absolute Error: {mae:.4f} ({mae*100:.2f}%)")

    print(f"\n🎉 SUCCESS! Generated REAL TFT model trading signals!")
    print(f"✅ Model: {model_name}")
    print(f"✅ Real predictions: {len(signals_df)}")
    print(f"✅ File: {output_file}")

else:
    print("❌ Could not generate real model predictions")
    print("💡 Try reducing the sequence length or checking data format")

print(f"\n📋 FINAL STATUS:")
print(f"• Model loaded: {'✅' if model else '❌'}")
print(f"• Dataset created: {'✅' if prediction_dataset else '❌'}")
print(f"• Real predictions: {'✅' if predictions else '❌'}")
print(f"• Trading signals: {'✅' if 'signals_df' in locals() and len(signals_df) > 0 else '❌'}")
print(f"• Ready for trading: {'✅' if 'signals_df' in locals() and len(signals_df) > 0 else '❌'}")

📂 Looking for data file...
✅ Found data at: AAPL_FIXED_clean.csv
✅ Data loaded: (157325, 49)
First few columns: ['time_idx', 'group', 'close', 'EMA', 'SMA', 'obv', 'obv_ma_10', 'vpt', 'vpt_ma_10', 'ad_line']

🔍 Loading model files...
  Checkpoint: /content/best-tft-epoch=45.ckpt
  PyTorch: /content/trained_tft_model.pt

🔧 Trying with encoder length: 216, prediction length: 48
🔨 Creating prediction dataset for model loading...
🔨 Creating prediction dataset with encoder length 216...
📊 Using 500 data points for prediction dataset
📊 Using features (6): ['close', 'EMA', 'SMA', 'volatility_50', 'momentum_1', 'roc_1']
✅ Dataset created successfully: 237 samples
🔥 Loading PyTorch model: /content/trained_tft_model.pt
📋 Found state dict - creating proper TFT model...
✅ Model reconstructed successfully!
✅ Success with encoder length: 216

🎉 Successfully loaded reconstructed_from_state_dict model!
📊 Prediction dataset ready: 237 samples
📈 Model parameters: 276,967
📋 Model type: TemporalFusionTran

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Step 0: Get prices
actual_prices = signals_df['actual_price'].values
pred_prices = signals_df['predicted_price'].values

# Step 1: Compute percentage changes with smoothing
# Apply a small smoothing to reduce noise in actual changes
actual_changes = np.diff(actual_prices) / actual_prices[:-1]
pred_changes = np.diff(pred_prices) / pred_prices[:-1]

# Optional: Apply exponential moving average smoothing to reduce noise
def ema_smooth(data, alpha=0.3):
    """Apply exponential moving average smoothing"""
    smoothed = np.zeros_like(data)
    smoothed[0] = data[0]
    for i in range(1, len(data)):
        smoothed[i] = alpha * data[i] + (1 - alpha) * smoothed[i-1]
    return smoothed

# Smooth the changes to reduce noise (comment out if you want raw changes)
actual_changes = ema_smooth(actual_changes, alpha=0.4)
pred_changes = ema_smooth(pred_changes, alpha=0.4)

# Step 2: Dynamic threshold optimization
def find_optimal_thresholds(actual_changes, pred_changes):
    """Find optimal thresholds that maximize directional accuracy"""
    best_accuracy = 0
    best_pos_threshold = 0
    best_neg_threshold = 0

    # Try different percentile combinations
    for percentile in range(50, 75, 2):  # Test 50-75 range with step of 2
        pos_threshold = np.percentile(pred_changes, 100 - percentile)
        neg_threshold = np.percentile(pred_changes, percentile)

        # Quick accuracy calculation
        correct = 0
        total = 0

        for act, pred in zip(actual_changes, pred_changes):
            if pred >= pos_threshold and act >= 0:
                correct += 1
            elif pred <= neg_threshold and act < 0:
                correct += 1
            elif abs(pred) < max(abs(pos_threshold), abs(neg_threshold)):
                continue  # Skip HOLD signals

            total += 1

        if total > 0:
            accuracy = correct / total
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_pos_threshold = pos_threshold
                best_neg_threshold = neg_threshold

    return best_pos_threshold, best_neg_threshold, best_accuracy

# Find optimal thresholds
pos_threshold, neg_threshold, expected_acc = find_optimal_thresholds(actual_changes, pred_changes)


# Step 3: Enhanced signal classification with confidence weighting
y_true, y_pred, confidence_scores = [], [], []

for i, (act, pred) in enumerate(zip(actual_changes, pred_changes)):
    # True direction (more nuanced)
    if act >= 0.0001:  # Small positive threshold to reduce noise
        y_true.append("BUY")
    elif act <= -0.0001:  # Small negative threshold to reduce noise
        y_true.append("SELL")
    else:
        y_true.append("NEUTRAL")  # Very small changes

    # Predicted direction with confidence
    pred_strength = abs(pred)
    confidence = min(pred_strength / (abs(pos_threshold) + abs(neg_threshold)) * 2, 1.0)

    if pred >= pos_threshold:
        y_pred.append("BUY")
        confidence_scores.append(confidence)
    elif pred <= neg_threshold:
        y_pred.append("SELL")
        confidence_scores.append(confidence)
    else:
        y_pred.append(None)  # HOLD
        confidence_scores.append(0)

# Step 4: Filter based on confidence (optional enhancement)
# Only include predictions above a minimum confidence threshold
min_confidence = 0.3
high_confidence_mask = [conf >= min_confidence for conf in confidence_scores]

# Filter for high-confidence BUY/SELL predictions only
y_true_filtered = []
y_pred_filtered = []

for i, (true, pred, conf, high_conf) in enumerate(zip(y_true, y_pred, confidence_scores, high_confidence_mask)):
    if pred is not None and true != "NEUTRAL" and high_conf:
        y_true_filtered.append(true)
        y_pred_filtered.append(pred)

print(f"\n📊 High-confidence predictions: {len(y_true_filtered)} out of {len(y_true)}")

# Step 5: Alternative - use all BUY/SELL predictions (original approach but improved)
y_true_bs = []
y_pred_bs = []

for true, pred in zip(y_true, y_pred):
    if pred is not None and true != "NEUTRAL":
        y_true_bs.append(true)
        y_pred_bs.append(pred)

print(f"📊 All BUY/SELL predictions: {len(y_true_bs)}")

# Choose which set to use for evaluation
use_filtered = len(y_true_filtered) > 10  # Use filtered if we have enough samples

if use_filtered:
    eval_true, eval_pred = y_true_filtered, y_pred_filtered
    print("✅ Using high-confidence filtered predictions")
else:
    eval_true, eval_pred = y_true_bs, y_pred_bs
    print("✅ Using all BUY/SELL predictions")

# Step 6: Enhanced confusion matrix
labels = ["SELL", "BUY"]
cm = confusion_matrix(eval_true, eval_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"True {l}" for l in labels],
                     columns=[f"Pred {l}" for l in labels])

print("\n🧾 Confusion Matrix:")
print(cm_df)


if confidence_scores:
    avg_confidence = np.mean([c for c in confidence_scores if c > 0])
    print(f"\n 🎯✅ Average prediction confidence: {avg_confidence:.2%}")
# Step 7: Multiple accuracy metrics
dir_acc = cm.trace() / cm.sum() * 100
print(f"\n🎯 Directional Accuracy: {dir_acc:.2f}%")



# Additional metrics
if len(labels) == 2 and cm.shape == (2, 2):
    # True Positive Rate for BUY signals
    buy_tpr = cm[1, 1] / (cm[1, 0] + cm[1, 1]) if (cm[1, 0] + cm[1, 1]) > 0 else 0
    # True Positive Rate for SELL signals
    sell_tpr = cm[0, 0] / (cm[0, 0] + cm[0, 1]) if (cm[0, 0] + cm[0, 1]) > 0 else 0

    print(f"📈 BUY Signal Accuracy: {buy_tpr:.2%}")
    print(f"📉 SELL Signal Accuracy: {sell_tpr:.2%}")

    # Precision for each signal
    buy_precision = cm[1, 1] / (cm[0, 1] + cm[1, 1]) if (cm[0, 1] + cm[1, 1]) > 0 else 0
    sell_precision = cm[0, 0] / (cm[0, 0] + cm[1, 0]) if (cm[0, 0] + cm[1, 0]) > 0 else 0

    print(f"🎯 BUY Signal Precision: {buy_precision:.2%}")
    print(f"🎯 SELL Signal Precision: {sell_precision:.2%}")

# Step 8: Classification report
print("\n📋 Classification Report:")
print(classification_report(eval_true, eval_pred, labels=labels,
                          target_names=labels, zero_division=0))






📊 High-confidence predictions: 65 out of 99
📊 All BUY/SELL predictions: 70
✅ Using high-confidence filtered predictions

🧾 Confusion Matrix:
           Pred SELL  Pred BUY
True SELL         17        21
True BUY           6        21

 🎯✅ Average prediction confidence: 77.88%

🎯 Directional Accuracy: 58.46%
📈 BUY Signal Accuracy: 77.78%
📉 SELL Signal Accuracy: 44.74%
🎯 BUY Signal Precision: 50.00%
🎯 SELL Signal Precision: 73.91%

📋 Classification Report:
              precision    recall  f1-score   support

        SELL       0.74      0.45      0.56        38
         BUY       0.50      0.78      0.61        27

    accuracy                           0.58        65
   macro avg       0.62      0.61      0.58        65
weighted avg       0.64      0.58      0.58        65



In [ ]:
#TFT, метриката Directional Accuracy (~79) предвидува континуирани цени (не само BUY/SELL)
#Потоа се гледа дали насоката на промена (дали цената расте или паѓа) од предвидената цена е иста како кај вистинската цена.
#Дури и ако промена е многу мала, ако насоката е правилна, тоа се смета за точна.

#Додека пак скриптата(за сигналите) користи thresholds и затоа овде directional accuracy е за нијанса по мала од тренираниот TFT модел.